# Модуль 3 · Урок 1 — Основи SQL та реляційні бази даних

👋 Досі наші дані «жили» лише поки працює програма. Тепер навчимося зберігати їх **надовго** —
у **базі даних**, і керувати ними мовою **SQL**. Це фундамент бекенду й одна з **найбільших тем
на співбесідах**.

**Передумова:** Модуль 1 (основи Python). Спеціальних знань не треба — SQL вивчаємо з нуля.

### Що ви вмітимете після уроку
- розуміти, що таке **реляційна БД**, таблиці, типи даних, обмеження;
- проєктувати **зв'язки** між таблицями (ключі, one-to-many, many-to-many);
- писати запити: `CREATE`, `INSERT`, `SELECT`, `UPDATE`, `DELETE`, `JOIN`, агрегації;
- користуватися транзакціями та захищатись від **SQL-ін'єкцій**.

> 🧪 **Ми пишемо справжній SQL прямо тут** — через вбудований модуль `sqlite3`. Нічого
> встановлювати не треба; кожен запит одразу виконується. Той самий SQL працює й у
> PostgreSQL/MySQL (відмінності позначу окремо).

> 📌 Позначка **🔎 Важливо знати** — теми, які майже завжди питають на співбесідах.

## 1. Що таке база даних. Реляційні vs нереляційні

**База даних (БД)** — організоване сховище даних. **СУБД** (система керування БД) — програма,
що цим сховищем керує (PostgreSQL, MySQL, SQLite, MongoDB…).

Два великі світи:

| | **Реляційні (SQL)** | **Нереляційні (NoSQL)** |
|---|---|---|
| Модель даних | **таблиці** (рядки й стовпці) | документи, ключ-значення, графи… |
| Схема | сувора (наперед задана) | гнучка |
| Мова | **SQL** | різні API |
| Приклади | PostgreSQL, MySQL, SQLite, Oracle | MongoDB, Redis, Cassandra |
| Коли обирати | пов'язані дані, важлива цілісність (гроші, замовлення) | великі обсяги, гнучка структура, кеш |

У цьому уроці — **реляційні БД і SQL** (найпоширеніший вибір для бекенду).

## 2. Структура реляційної БД: таблиці

Дані зберігаються в **таблицях**. Таблиця — як аркуш Excel:
- **стовпець (column / поле)** — характеристика (напр. `name`, `age`);
- **рядок (row / запис)** — один об'єкт (один користувач).

```
              таблиця users
   ┌────┬──────────┬─────┬────────────────┐
   │ id │ name     │ age │ email          │  <- стовпці (поля)
   ├────┼──────────┼─────┼────────────────┤
   │ 1  │ Ірина    │ 25  │ iryna@mail.com │  <- рядок (запис)
   │ 2  │ Олег     │ 40  │ oleg@mail.com  │
   └────┴──────────┴─────┴────────────────┘
```

## 3. Готуємось: підключення до SQLite

Створюємо базу **в пам'яті** (`:memory:`) — вона існує, поки працює зошит. Також напишемо
маленький помічник `show()`, щоб гарно друкувати результати запитів (це просто зручність, не SQL).

In [2]:
import sqlite3

conn = sqlite3.connect(":memory:")       # база в пам'яті
conn.isolation_level = None              # автокоміт: даємо змогу керувати транзакціями вручну (BEGIN/COMMIT)
conn.execute("PRAGMA foreign_keys = ON") # увімкнути перевірку зовнішніх ключів (про них — далі)

def show(sql, params=()):
    """Виконати запит; якщо це SELECT — надрукувати результат таблицею."""
    cur = conn.execute(sql, params)
    if cur.description:                          # є стовпці -> це SELECT
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        w = [len(c) for c in cols]
        for r in rows:
            for i, v in enumerate(r):
                w[i] = max(w[i], len(str(v)))
        fmt = lambda vals: " | ".join(str(v).ljust(w[i]) for i, v in enumerate(vals))
        print(fmt(cols))
        print("-+-".join("-" * x for x in w))
        for r in rows:
            print(fmt(r))
    conn.commit()

print("Готово: підключено до SQLite", sqlite3.sqlite_version)

Готово: підключено до SQLite 3.51.0


## 4. Типи даних

Кожен стовпець має **тип** — які значення в ньому дозволені. Основні типи (стандартний SQL):

| Тип | Що зберігає | Приклад |
|---|---|---|
| `INTEGER` / `INT` | цілі числа | `42` |
| `REAL` / `FLOAT` / `NUMERIC` | дробові | `19.99` |
| `TEXT` / `VARCHAR(n)` | текст | `'Ірина'` |
| `BOOLEAN` | так/ні | `TRUE` / `FALSE` |
| `DATE` / `TIMESTAMP` | дата / дата-час | `'2026-07-08'` |
| `BLOB` | бінарні дані | файл, картинка |

> 🔎 **Важливо знати про діалекти.** SQLite має спрощену систему типів (INTEGER, REAL, TEXT,
> BLOB, NULL) і досить «гнучкий». PostgreSQL і MySQL — **суворіші** й багатші на типи
> (`VARCHAR(255)`, `SERIAL`, `TIMESTAMP`, `DECIMAL`…). У Модулі 3, Урок 2 працюватимемо з
> PostgreSQL, де це важливіше.

## 5. `CREATE TABLE` — створюємо таблицю (DDL)

**DDL** (Data Definition Language) — команди, що описують **структуру** БД. Головна — `CREATE TABLE`.
Синтаксис: перелічуємо стовпці та їхні типи.

In [3]:
conn.execute("""
CREATE TABLE users (
    id    INTEGER PRIMARY KEY,   -- унікальний номер запису (детально далі)
    name  TEXT,
    age   INTEGER,
    email TEXT
)
""")
print("Таблицю users створено")

Таблицю users створено


## 6. Обмеження (constraints)

**Обмеження** — правила, які БД **сама** перевіряє, щоб дані завжди були коректні. Головні:

- **`PRIMARY KEY`** — унікальний ідентифікатор рядка (не повторюється, не порожній).
- **`NOT NULL`** — значення обов'язкове (не можна лишити порожнім).
- **`UNIQUE`** — значення не може повторюватись (напр. email).
- **`DEFAULT`** — значення за замовчуванням, якщо не вказали.
- **`CHECK`** — власна умова (напр. вік не від'ємний).

Перестворимо `users` з обмеженнями:

In [4]:
conn.execute("DROP TABLE users")     # видалимо стару, створимо кращу
conn.execute("""
CREATE TABLE users (
    id    INTEGER PRIMARY KEY,
    name  TEXT    NOT NULL,                       -- обов'язкове
    age   INTEGER CHECK (age >= 0),               -- не може бути від'ємним
    email TEXT    UNIQUE,                          -- не повторюється
    city  TEXT    DEFAULT 'Львів'                  -- за замовчуванням
)
""")
print("users із обмеженнями створено")

users із обмеженнями створено


In [5]:
# Перевіримо, що обмеження РЕАЛЬНО працюють:
# 1) NOT NULL — ім'я обов'язкове
try:
    conn.execute("INSERT INTO users (id, name) VALUES (1, NULL)")
except sqlite3.IntegrityError as e:
    print("NOT NULL спрацював:", e)

# 2) CHECK — вік не може бути від'ємним
try:
    conn.execute("INSERT INTO users (id, name, age) VALUES (1, 'Тест', -5)")
except sqlite3.IntegrityError as e:
    print("CHECK спрацював:", e)

NOT NULL спрацював: NOT NULL constraint failed: users.name
CHECK спрацював: CHECK constraint failed: age >= 0


## 7. `INSERT` — додаємо дані (DML)

**DML** (Data Manipulation Language) — команди для роботи з **даними**. `INSERT` додає рядки.

In [6]:
conn.execute("INSERT INTO users (id, name, age, email, city) VALUES (1, 'Ірина', 25, 'iryna@mail.com', 'Київ')")
conn.execute("INSERT INTO users (id, name, age, email) VALUES (2, 'Олег', 40, 'oleg@mail.com')")  # city візьметься за замовчуванням

# Кілька рядків одразу:
conn.executemany(
    "INSERT INTO users (id, name, age, email, city) VALUES (?, ?, ?, ?, ?)",
    [(3, 'Марія', 30, 'maria@mail.com', 'Львів'),
     (4, 'Богдан', 22, 'bohdan@mail.com', 'Одеса'),
     (5, 'Олена', 35, 'olena@mail.com', 'Київ')]
)
conn.commit()
print("Дані додано")
show("SELECT * FROM users")

Дані додано
id | name   | age | email           | city 
---+--------+-----+-----------------+------
1  | Ірина  | 25  | iryna@mail.com  | Київ 
2  | Олег   | 40  | oleg@mail.com   | Львів
3  | Марія  | 30  | maria@mail.com  | Львів
4  | Богдан | 22  | bohdan@mail.com | Одеса
5  | Олена  | 35  | olena@mail.com  | Київ 


## 8. `SELECT` — читаємо дані

`SELECT` — найважливіша команда: дістає дані з таблиці. Загальний вигляд:

```sql
SELECT стовпці FROM таблиця WHERE умова ORDER BY стовпець;
```

In [7]:
# Усі стовпці всіх рядків:
show("SELECT * FROM users")

id | name   | age | email           | city 
---+--------+-----+-----------------+------
1  | Ірина  | 25  | iryna@mail.com  | Київ 
2  | Олег   | 40  | oleg@mail.com   | Львів
3  | Марія  | 30  | maria@mail.com  | Львів
4  | Богдан | 22  | bohdan@mail.com | Одеса
5  | Олена  | 35  | olena@mail.com  | Київ 


In [8]:
# Лише певні стовпці:
show("SELECT name, city FROM users")

name   | city 
-------+------
Ірина  | Київ 
Олег   | Львів
Марія  | Львів
Богдан | Одеса
Олена  | Київ 


### `WHERE` — фільтрація рядків
Залишає лише рядки, що відповідають умові. Оператори: `=`, `!=`, `<`, `>`, `<=`, `>=`,
`AND`, `OR`, `IN`, `BETWEEN`, `LIKE` (пошук за шаблоном).

In [9]:
print("--- старші за 28 ---")
show("SELECT name, age FROM users WHERE age > 28")

print("--- з Києва ---")
show("SELECT name, city FROM users WHERE city = 'Київ'")

print("--- вік від 25 до 35 ---")
show("SELECT name, age FROM users WHERE age BETWEEN 25 AND 35")

print("--- ім'я починається на 'О' ---")
show("SELECT name FROM users WHERE name LIKE 'О%'")

--- старші за 28 ---
name  | age
------+----
Олег  | 40 
Марія | 30 
Олена | 35 
--- з Києва ---
name  | city
------+-----
Ірина | Київ
Олена | Київ
--- вік від 25 до 35 ---
name  | age
------+----
Ірина | 25 
Марія | 30 
Олена | 35 
--- ім'я починається на 'О' ---
name 
-----
Олег 
Олена


### `ORDER BY` — сортування
`ASC` — за зростанням (за замовчуванням), `DESC` — за спаданням.

In [10]:
print("--- за віком, від старших ---")
show("SELECT name, age FROM users ORDER BY age DESC")

print("--- за іменем (А-Я) ---")
show("SELECT name FROM users ORDER BY name ASC")

--- за віком, від старших ---
name   | age
-------+----
Олег   | 40 
Олена  | 35 
Марія  | 30 
Ірина  | 25 
Богдан | 22 
--- за іменем (А-Я) ---
name  
------
Ірина 
Богдан
Марія 
Олег  
Олена 


## 9. Зв'язки між таблицями: `PRIMARY KEY` та `FOREIGN KEY`

Реальні дані пов'язані: у користувача є замовлення, у замовлення — товари. Замість того щоб
дублювати все в одній таблиці, дані **розкладають** по таблицях і **зв'язують ключами**.

- **`PRIMARY KEY` (первинний ключ)** — унікальний ідентифікатор рядка в **своїй** таблиці.
- **`FOREIGN KEY` (зовнішній ключ)** — стовпець, що **посилається** на `PRIMARY KEY` іншої таблиці.

Створимо таблицю `orders`, де кожне замовлення належить користувачу:

In [11]:
conn.execute("""
CREATE TABLE orders (
    id       INTEGER PRIMARY KEY,
    user_id  INTEGER NOT NULL,
    product  TEXT    NOT NULL,
    amount   REAL,
    FOREIGN KEY (user_id) REFERENCES users(id)   -- зв'язок з users
)
""")

conn.executemany(
    "INSERT INTO orders (id, user_id, product, amount) VALUES (?, ?, ?, ?)",
    [(1, 1, 'Клавіатура', 800),
     (2, 1, 'Миша', 400),
     (3, 3, 'Монітор', 6000),
     (4, 5, 'Ноутбук', 30000)]
)
conn.commit()
show("SELECT * FROM orders")

id | user_id | product    | amount 
---+---------+------------+--------
1  | 1       | Клавіатура | 800.0  
2  | 1       | Миша       | 400.0  
3  | 3       | Монітор    | 6000.0 
4  | 5       | Ноутбук    | 30000.0


In [12]:
# FOREIGN KEY захищає цілісність: не можна створити замовлення для НЕіснуючого користувача
try:
    conn.execute("INSERT INTO orders (id, user_id, product) VALUES (99, 999, 'Привид')")
except sqlite3.IntegrityError as e:
    print("FOREIGN KEY спрацював:", e)

FOREIGN KEY спрацював: FOREIGN KEY constraint failed


### Типи зв'язків

```
one-to-many (один-до-багатьох)      -- НАЙЧАСТІШИЙ
   users (1) ───< orders (багато)
   Один користувач має багато замовлень; кожне замовлення — одного користувача.
   (з боку замовлення це "many-to-one", багато-до-одного — той самий зв'язок, інший погляд)

one-to-one (один-до-одного)
   users (1) ─── profiles (1)
   У користувача рівно один профіль. FK + UNIQUE на боці профілю.

many-to-many (багато-до-багатьох)
   students (багато) >───< courses (багато)
   Студент вивчає багато курсів, курс має багато студентів.
   Реалізують через ТРЕТЮ таблицю-зв'язок (junction), напр. enrollments(student_id, course_id).
```

Зробимо приклад **many-to-many** — студенти й курси через таблицю-зв'язок:

In [13]:
conn.execute("CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT)")
conn.execute("CREATE TABLE courses  (id INTEGER PRIMARY KEY, title TEXT)")
conn.execute("""
CREATE TABLE enrollments (
    student_id INTEGER,
    course_id  INTEGER,
    FOREIGN KEY (student_id) REFERENCES students(id),
    FOREIGN KEY (course_id)  REFERENCES courses(id)
)
""")

conn.executemany("INSERT INTO students VALUES (?, ?)", [(1, 'Аня'), (2, 'Богдан')])
conn.executemany("INSERT INTO courses  VALUES (?, ?)", [(1, 'Python'), (2, 'SQL')])
# Аня вивчає обидва курси, Богдан — тільки SQL:
conn.executemany("INSERT INTO enrollments VALUES (?, ?)", [(1, 1), (1, 2), (2, 2)])
conn.commit()

show("""
SELECT s.name AS студент, c.title AS курс
FROM enrollments e
JOIN students s ON s.id = e.student_id
JOIN courses  c ON c.id = e.course_id
""")

студент | курс  
--------+-------
Аня     | Python
Аня     | SQL   
Богдан  | SQL   


## 10. Нормалізація та денормалізація

**Нормалізація** — розкладання даних по таблицях так, щоб **уникнути дублювання** й аномалій.
Коротко про перші три «нормальні форми» (НФ):

- **1НФ:** кожна клітинка містить **одне** значення (жодних списків у полі).
- **2НФ:** 1НФ + кожне поле залежить від **усього** первинного ключа.
- **3НФ:** 2НФ + немає полів, що залежать від **інших неключових** полів.

**Приклад проблеми (не нормалізовано):** якщо в таблиці `orders` тримати ще й `user_name` та
`user_city`, то ім'я користувача **дублюється** в кожному його замовленні. Змінив місто — треба
оновлювати всюди (аномалія оновлення). **Рішення:** винести користувача в окрему таблицю `users`
і посилатися через `user_id` — саме так ми й зробили.

**Денормалізація** — свідоме **зворотне** дублювання даних заради **швидкості читання** (менше
`JOIN`). Плата — складніші оновлення й ризик неузгодженості. Застосовують точково, коли
продуктивність важливіша (напр. аналітика, кеш).

## 11. `ALTER` та `DROP` (+ `CASCADE`)

- **`ALTER TABLE`** — змінити структуру наявної таблиці (додати/перейменувати стовпець).
- **`DROP TABLE`** — видалити таблицю повністю.
- **`CASCADE`** — «каскадне» видалення пов'язаних даних.

In [14]:
# ALTER: додати новий стовпець
conn.execute("ALTER TABLE users ADD COLUMN phone TEXT")
show("SELECT id, name, phone FROM users LIMIT 3")

id | name  | phone
---+-------+------
1  | Ірина | None 
2  | Олег  | None 
3  | Марія | None 


### 🔎 Важливо знати: `ON DELETE CASCADE`

Що станеться із замовленнями, якщо видалити користувача? За замовчуванням БД **не дасть**
видалити (є посилання). Якщо оголосити зовнішній ключ із **`ON DELETE CASCADE`**, то разом із
користувачем **автоматично** видаляться і його замовлення:

```sql
FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE
```

`DROP TABLE ... CASCADE` (у PostgreSQL) так само видаляє залежні об'єкти (напр. представлення).
> ⚠️ Обережно: `CASCADE` видаляє дані **без попередження**.

## 12. `UPDATE` та `DELETE`

- **`UPDATE`** — змінити наявні рядки.
- **`DELETE`** — видалити рядки.

> ⚠️ **Найнебезпечніша помилка:** забути `WHERE`! `UPDATE users SET city='X'` **без** `WHERE`
> змінить **усі** рядки. Так само `DELETE FROM users` без `WHERE` видалить **усе**.

In [15]:
# UPDATE з WHERE — змінюємо місто одному користувачу
conn.execute("UPDATE users SET city = 'Харків' WHERE id = 2")
show("SELECT id, name, city FROM users WHERE id = 2")

id | name | city  
---+------+-------
2  | Олег | Харків


In [16]:
# DELETE з WHERE — видаляємо один рядок
conn.execute("DELETE FROM users WHERE id = 4")
conn.commit()
show("SELECT id, name FROM users")

id | name 
---+------
1  | Ірина
2  | Олег 
3  | Марія
5  | Олена


## 13. Агрегація: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`

**Агрегатні функції** рахують одне значення з багатьох рядків:
- `COUNT()` — кількість, `SUM()` — сума, `AVG()` — середнє, `MIN()`/`MAX()` — мінімум/максимум.

`GROUP BY` — рахувати **по групах**; `HAVING` — фільтр **для груп** (як `WHERE`, але після агрегації).

In [17]:
print("--- скільки користувачів усього ---")
show("SELECT COUNT(*) AS кількість FROM users")

print("--- середній, мін і макс вік ---")
show("SELECT AVG(age) AS середній, MIN(age) AS наймолодший, MAX(age) AS найстарший FROM users")

print("--- сума замовлень ---")
show("SELECT SUM(amount) AS всього_грн FROM orders")

--- скільки користувачів усього ---
кількість
---------
4        
--- середній, мін і макс вік ---
середній | наймолодший | найстарший
---------+-------------+-----------
32.5     | 25          | 40        
--- сума замовлень ---
всього_грн
----------
37200.0   


In [18]:
# GROUP BY — скільки користувачів у кожному місті:
show("SELECT city, COUNT(*) AS людей FROM users GROUP BY city")

city   | людей
-------+------
Київ   | 2    
Львів  | 1    
Харків | 1    


In [19]:
# HAVING — залишити тільки міста, де більше за 1 користувача:
show("""
SELECT city, COUNT(*) AS людей
FROM users
GROUP BY city
HAVING COUNT(*) > 1
""")

city | людей
-----+------
Київ | 2    


## 14. `JOIN` — об'єднання таблиць

`JOIN` дозволяє в одному запиті брати дані **з кількох таблиць**, поєднуючи їх за спільним
стовпцем (зазвичай `foreign key = primary key`).

```
   users                 orders
   id name          id user_id product
   1  Ірина         1  1       Клавіатура
   2  Олег          2  1       Миша
   3  Марія         3  3       Монітор

   INNER JOIN users.id = orders.user_id  -> лише ті, у кого Є замовлення
```

Види з'єднань:
- **`INNER JOIN`** — лише збіги в **обох** таблицях.
- **`LEFT JOIN`** — усі з лівої + збіги з правої (де немає — `NULL`).
- **`RIGHT JOIN`** — усі з правої + збіги з лівої.
- **`FULL OUTER JOIN`** — усе з обох таблиць.
- **`CROSS JOIN`** — усі можливі пари (декартів добуток).

In [20]:
# INNER JOIN: користувачі, у яких Є замовлення (+ що саме)
show("""
SELECT u.name, o.product, o.amount
FROM users u
INNER JOIN orders o ON o.user_id = u.id
""")

name  | product    | amount 
------+------------+--------
Ірина | Клавіатура | 800.0  
Ірина | Миша       | 400.0  
Марія | Монітор    | 6000.0 
Олена | Ноутбук    | 30000.0


In [21]:
# LEFT JOIN: УСІ користувачі, навіть без замовлень (там буде NULL)
show("""
SELECT u.name, o.product
FROM users u
LEFT JOIN orders o ON o.user_id = u.id
ORDER BY u.name
""")

name  | product   
------+-----------
Ірина | Клавіатура
Ірина | Миша      
Марія | Монітор   
Олег  | None      
Олена | Ноутбук   


In [22]:
# RIGHT / FULL OUTER JOIN (підтримуються сучасним SQLite і PostgreSQL):
print("--- RIGHT JOIN ---")
show("SELECT u.name, o.product FROM users u RIGHT JOIN orders o ON o.user_id = u.id")

print("--- FULL OUTER JOIN ---")
show("SELECT u.name, o.product FROM users u FULL OUTER JOIN orders o ON o.user_id = u.id")

--- RIGHT JOIN ---
name  | product   
------+-----------
Ірина | Клавіатура
Ірина | Миша      
Марія | Монітор   
Олена | Ноутбук   
--- FULL OUTER JOIN ---
name  | product   
------+-----------
Ірина | Клавіатура
Ірина | Миша      
Олег  | None      
Марія | Монітор   
Олена | Ноутбук   


In [23]:
# CROSS JOIN: усі можливі пари (обережно — рядків = N * M)
show("SELECT s.name, c.title FROM students s CROSS JOIN courses c")

name   | title 
-------+-------
Аня    | Python
Аня    | SQL   
Богдан | Python
Богдан | SQL   


## 15. `VIEW` — збережений запит (представлення)

**`VIEW`** — це «віртуальна таблиця»: збережений `SELECT`, до якого можна звертатись за іменем,
ніби це таблиця. Зручно ховати складні запити за простим іменем.

In [24]:
conn.execute("""
CREATE VIEW user_orders AS
SELECT u.name AS user, o.product, o.amount
FROM users u
JOIN orders o ON o.user_id = u.id
""")

# Тепер звертаємось до view як до таблиці:
show("SELECT * FROM user_orders WHERE amount > 500")

user  | product    | amount 
------+------------+--------
Ірина | Клавіатура | 800.0  
Марія | Монітор    | 6000.0 
Олена | Ноутбук    | 30000.0


## 16. Функції та процедури

**Функції** повертають значення й використовуються в запитах. Багато **вбудованих** функцій є
в кожній БД (працюють і в SQLite):

In [25]:
show("SELECT UPPER(name) AS великими, LENGTH(name) AS довжина FROM users LIMIT 3")
show("SELECT ROUND(AVG(age), 1) AS середній_вік FROM users")

великими | довжина
---------+--------
Ірина    | 5      
Олег     | 4      
Марія    | 5      
середній_вік
------------
32.5        


### 🔎 Важливо знати: збережені процедури

**Збережена процедура (stored procedure)** — це іменований блок SQL-коду з логікою (умови, цикли),
що зберігається **в самій БД** і викликається за іменем. Це фішка «дорослих» СУБД (PostgreSQL,
MySQL); **у SQLite їх немає**.

Приклад синтаксису **PostgreSQL** (запустимо в Уроці 2, коли підключимось до Postgres):
```sql
CREATE PROCEDURE give_bonus(user_id INT)
LANGUAGE plpgsql
AS $$
BEGIN
    UPDATE accounts SET balance = balance + 100 WHERE id = user_id;
END;
$$;

CALL give_bonus(1);
```
> Функції vs процедури: **функція повертає значення** й зазвичай викликається в `SELECT`;
> **процедура виконує дії** (напр. кілька `UPDATE`) і викликається через `CALL`.

## 17. Транзакції

**Транзакція** — група операцій, які виконуються **як одне ціле**: або **всі** успішно
(`COMMIT`), або **жодна** (`ROLLBACK`). Класичний приклад — переказ грошей: списати з одного
рахунку й зарахувати на інший **мають статися разом**.

- `BEGIN` — почати транзакцію;
- `COMMIT` — підтвердити (зберегти зміни);
- `ROLLBACK` — скасувати **всі** зміни транзакції.

In [26]:
conn.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, owner TEXT, balance REAL)")
conn.executemany("INSERT INTO accounts VALUES (?, ?, ?)", [(1, 'Аня', 1000), (2, 'Богдан', 500)])
conn.commit()

# Переказ 300 від Ані до Богдана — обидві дії в одній транзакції
try:
    conn.execute("BEGIN")
    conn.execute("UPDATE accounts SET balance = balance - 300 WHERE id = 1")
    conn.execute("UPDATE accounts SET balance = balance + 300 WHERE id = 2")
    conn.execute("COMMIT")
    print("Переказ успішний")
except Exception:
    conn.execute("ROLLBACK")
    print("Помилка — усе скасовано")

show("SELECT owner, balance FROM accounts")

Переказ успішний
owner  | balance
-------+--------
Аня    | 700.0  
Богдан | 800.0  


In [27]:
# ROLLBACK у дії: почали змінювати, але передумали — дані НЕ змінились
conn.execute("BEGIN")
conn.execute("UPDATE accounts SET balance = 0 WHERE id = 1")
conn.execute("ROLLBACK")     # скасовуємо
show("SELECT owner, balance FROM accounts WHERE id = 1")   # баланс на місці

owner | balance
------+--------
Аня   | 700.0  


> 🔎 Надійність транзакцій описують властивості **ACID** — детально розберемо в **Уроці 2**.

## 18. 🔎 Важливо знати: SQL-ін'єкції

**SQL-ін'єкція** — одна з найнебезпечніших вразливостей. Вона виникає, коли дані від користувача
**вклеюють** прямо в текст запиту. Зловмисник може підсунути шматок SQL і **обійти перевірку**
чи знищити дані.

Розглянемо «логін» — небезпечний спосіб (склеювання рядків):

In [28]:
# ❌ НЕБЕЗПЕЧНО: підставляємо ввід користувача прямо в SQL
def login_unsafe(name):
    sql = "SELECT * FROM users WHERE name = '" + name + "'"
    print("  виконується SQL:", sql)
    return conn.execute(sql).fetchall()

print("Звичайний вхід:")
print(" ", login_unsafe("Ірина"))

print("\nАтака — підставляємо \"' OR '1'='1\":")
rows = login_unsafe("' OR '1'='1")
print("  повернуло рядків:", len(rows), "-> зловмисник отримав ВСІХ користувачів!")

Звичайний вхід:
  виконується SQL: SELECT * FROM users WHERE name = 'Ірина'
  [(1, 'Ірина', 25, 'iryna@mail.com', 'Київ', None)]

Атака — підставляємо "' OR '1'='1":
  виконується SQL: SELECT * FROM users WHERE name = '' OR '1'='1'
  повернуло рядків: 4 -> зловмисник отримав ВСІХ користувачів!


Що сталося: введення `' OR '1'='1` перетворило умову на `name = '' OR '1'='1'`, яка
**завжди істинна** — і запит повернув усі рядки. Так обходять паролі й викрадають дані.

**✅ Захист — параметризовані запити.** Значення передають **окремо** (через `?` або `%s`),
і БД трактує їх **лише як дані**, ніколи як код:

In [29]:
# ✅ БЕЗПЕЧНО: значення йде окремим параметром, а не в текст запиту
def login_safe(name):
    return conn.execute("SELECT * FROM users WHERE name = ?", (name,)).fetchall()

print("Звичайний вхід:", len(login_safe("Ірина")), "рядок")
print("Спроба атаки:  ", len(login_safe("' OR '1'='1")), "рядків -> ін'єкція НЕ спрацювала ✅")

Звичайний вхід: 1 рядок
Спроба атаки:   0 рядків -> ін'єкція НЕ спрацювала ✅


> 🛡️ **Правило на все життя:** **ніколи** не склеюйте користувацький ввід у SQL рядками.
> Завжди використовуйте **параметризовані запити** (`?` у sqlite3, `%s` у psycopg2, або ORM).
> Це — обов'язкове питання на співбесідах із безпеки/бекенду.

# ✅ Підсумок уроку

- **Реляційна БД** зберігає дані в **таблицях** (рядки/стовпці) зі суворою схемою; мова — **SQL**.
- **Типи** й **обмеження** (`PRIMARY KEY`, `NOT NULL`, `UNIQUE`, `CHECK`, `DEFAULT`) тримають дані коректними.
- **Ключі** й **зв'язки:** `FOREIGN KEY` пов'язує таблиці; one-to-many найчастіший, many-to-many — через таблицю-зв'язок.
- **Нормалізація** прибирає дублювання; **денормалізація** — свідоме дублювання заради швидкості.
- **DDL:** `CREATE` / `ALTER` / `DROP` (+ `CASCADE`). **DML:** `INSERT` / `SELECT` / `UPDATE` / `DELETE`.
- **`SELECT`:** `WHERE`, `ORDER BY`, `GROUP BY` + агрегати (`COUNT`/`SUM`/`AVG`/`MIN`/`MAX`), `HAVING`.
- **`JOIN`:** inner / left / right / full / cross. **`VIEW`** — збережений запит.
- **Транзакції** (`BEGIN`/`COMMIT`/`ROLLBACK`) — все або нічого.
- **SQL-ін'єкції:** ніколи не склеюйте ввід у запит — лише **параметризовані запити**.

### 📝 Домашнє завдання
[domashnie-zavdannia-lektsiya-1.ipynb](domashnie-zavdannia-lektsiya-1.ipynb)

### ▶️ Далі
Урок 2 — **бази даних із Python**: ACID, індекси та оптимізація, `psycopg2`, **SQLAlchemy**,
CRUD, `alembic`, порівняння SQLite/MySQL/PostgreSQL.

### 📚 Хочу знати більше
- Інтерактивний тренажер SQL (укр./англ.): <https://sqlbolt.com/>
- Практика запитів: <https://pgexercises.com/>
- Документація SQLite: <https://www.sqlite.org/lang.html>
- PostgreSQL туторіал: <https://www.postgresqltutorial.com/>
- Про SQL-ін'єкції (OWASP): <https://owasp.org/www-community/attacks/SQL_Injection>